# Rule-Based Evaluation Pipeline

In [6]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv('Complaints.csv')
df = df[df['complaint_date'] != 'complaint_date'].copy()

orders_df = pd.read_csv('Orders.csv')
df['complaint_date'] = pd.to_datetime(df['complaint_date'], errors='coerce')
df = df.sort_values(by=['customer_id', 'complaint_date']).reset_index(drop=True)
df['prior_complaints'] = df.groupby('customer_id').cumcount()
order_counts = orders_df.groupby('customer_id').size().to_dict()
df['total_orders'] = df['customer_id'].map(order_counts).fillna(0)
df['complaint_ratio'] = (df['prior_complaints'] + 1) / df['total_orders'].apply(lambda x: 1 if x == 0 else x)
test_df = df.sample(frac=0.2).copy()

In [7]:
test_unlabeled = test_df[['complaint_id', 'complaint_text', 'customer_id', 'prior_complaints', 'complaint_ratio', 'total_orders']].copy()
test_unlabeled.sample(5)

,complaint_id,complaint_text,customer_id,prior_complaints,complaint_ratio,total_orders
73,CMP-302,screen ghost touch happening randomly,C-014,4,0.625,8
363,CMP-123,laptop overheating continuously,C-072,1,0.250,8
163,CMP-249,not same as picture online,C-031,3,0.500,8
51,CMP-164,wrong item delivered again 😡,C-010,2,0.375,8
411,CMP-175,earbuds low volume issue,C-082,1,0.250,8


In [8]:
def compute_fraud_features(row):
    text = str(row['complaint_text']).lower()
    score = 6
    classification = "Legitimate"
    
    if 'box' in text and any(w in text for w in ['random', 'broken', 'scratches', 'tampered', 'older', 'different']):
        score, classification = 95, 'Fraud'
    elif 'ordered' in text and ('got' in text or 'looks' in text or 'why' in text):
        score, classification = 93, 'Fraud'
    elif 'used' in text and ('already' in text or 'returning' in text or 'few' in text or 'event' in text or 'now' in text):
        score, classification = 85, 'Fraud'
    elif any(phrase in text for phrase in ['wrong item', 'wrong model', 'different from', 'not matching', 'match description']): 
        score, classification = 90, 'Fraud'
    elif any(phrase in text for phrase in ['changed my mind', 'dont like', "don't like", 'prefer', 'not my style', 'expected better', 'not what i expected', 'feels cheap', 'not worth']):
        score, classification = 18, 'Legitimate'
    else:
        words = text.split()
        if text.strip() in ['ok', 'bad product', 'not good', 'worst', 'bad', 'meh'] or (len(words) <= 3 and 'dead' not in text and 'not working' not in text):
            score, classification = 22, 'Legitimate'



    prior_complaints = row['prior_complaints']
    ratio = row['complaint_ratio']
    
    if prior_complaints >= 3:
        score += 15
        classification = "Fraud"
        
    if ratio > 0.5 and row['total_orders'] > 2:
        score += 10
        classification = "Fraud"

    if ratio <= 0.1 and prior_complaints == 0:
        score -= 5  
        
   
    score = max(0, min(100, score))
    if score >= 80:
        classification = "Fraud"
        
    return classification


test_unlabeled['Predicted_Fraud_Class'] = test_unlabeled.apply(compute_fraud_features, axis=1)
test_unlabeled.head()

,complaint_id,complaint_text,customer_id,prior_complaints,complaint_ratio,total_orders,Predicted_Fraud_Class
245,CMP-439,meh not great honestly,C-047,3,0.500,8,Fraud
135,CMP-394,mouse double click issue,C-026,2,0.375,8,Legitimate
55,CMP-197,monitor blank screen,C-011,1,0.250,8,Legitimate
180,CMP-429,this doesnt match description exactly,C-034,5,0.750,8,Fraud
250,CMP-049,I got wrong earbuds model,C-049,0,0.125,8,Legitimate


In [9]:

accuracy = accuracy_score(test_df['fraud_classification'], test_unlabeled['Predicted_Fraud_Class'])

print(f" Accuracy: {accuracy * 100:.2f}%")
print("\nOverall Report:\n")
print(classification_report(test_df['fraud_classification'], test_unlabeled['Predicted_Fraud_Class']))

 Accuracy: 64.00%

Overall Report:

              precision    recall  f1-score   support

       Fraud       0.58      0.64      0.61        44
  Legitimate       0.69      0.64      0.67        56

    accuracy                           0.64       100
   macro avg       0.64      0.64      0.64       100
weighted avg       0.64      0.64      0.64       100



In [10]:
from sklearn.metrics import accuracy_score, classification_report

def heuristic_dependency_parse(text, window=3):
    words = str(text).lower().split()
    dependencies = []
    
    targets = {'box', 'screen', 'item', 'model', 'audio', 'device', 'touch'}
    modifiers = {'broken', 'scratches', 'tampered', 'older', 'different', 'wrong', 'low', 'randomly', 'dead', 'rebooting'}
    
    for i, word in enumerate(words):
        if word in targets:
            start = max(0, i - window)
            end = min(len(words), i + window + 1)
            
            for j in range(start, end):
                if words[j] in modifiers:
                    if j > 0 and words[j-1] in ['not', 'no', 'never']:
                        dependencies.append((word, 'not_' + words[j]))
                    elif i != j:
                        dependencies.append((word, words[j]))
    return dependencies

def compute_fraud_features_dep(row):
    text = str(row['complaint_text']).lower()
    score = 6
    classification = "Legitimate"
    
    deps = heuristic_dependency_parse(text)
    
    if ('box', 'tampered') in deps or ('box', 'different') in deps:
        score, classification = 95, 'Fraud'
    elif ('item', 'wrong') in deps or ('model', 'wrong') in deps:
        score, classification = 90, 'Fraud'
    elif any((t, 'broken') in deps for t in ['screen', 'device']):
        score, classification = 85, 'Fraud'
    elif any(phrase in text for phrase in ['changed my mind', 'dont like', "don't like", 'prefer', 'not my style']):
        score, classification = 18, 'Legitimate' 
        
    prior_complaints = row['prior_complaints']
    ratio = row['complaint_ratio']
    
    if prior_complaints >= 3:
        score += 15
        classification = "Fraud"
    if ratio > 0.5 and row['total_orders'] > 2:
        score += 10
        classification = "Fraud"
    if ratio <= 0.1 and prior_complaints == 0:
        score -= 5  

    score = max(0, min(100, score))
    if score >= 80:
        classification = "Fraud"
        
    return classification

test_unlabeled['Dep_Predicted_Fraud'] = test_unlabeled.apply(compute_fraud_features_dep, axis=1)

dep_acc = accuracy_score(test_df['fraud_classification'], test_unlabeled['Dep_Predicted_Fraud'])
print(f"Dependency Parser Accuracy: {dep_acc * 100:.2f}%\n")
print(classification_report(test_df['fraud_classification'], test_unlabeled['Dep_Predicted_Fraud']))


Dependency Parser Accuracy: 56.00%

              precision    recall  f1-score   support

       Fraud       0.50      0.45      0.48        44
  Legitimate       0.60      0.64      0.62        56

    accuracy                           0.56       100
   macro avg       0.55      0.55      0.55       100
weighted avg       0.56      0.56      0.56       100

